<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.9-production-incidents/notebooks/exercises-10_9.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.9: Handling Production Incidents

*Module 10 · AI System Architecture & Production Deployment*

> Your AI chatbot starts recommending competitors. Your cost dashboard shows $500 spent in 2 hours. Your RAG pipeline returns gibberish. This lesson gives you the playbooks, rollback commands, and post-mortem templates to handle real AI incidents — fast and without panic.

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-10.9.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Severity Classification — How Bad Is It? — `incident_classifier.py`
2. Step 2 — Incident Response Playbooks — What to Do for Each Failure Mode — `playbooks.py`
3. Step 3 — Rollback Strategies — Coded Commands for Every Layer — `rollback.sh`
4. Step 4 — Post-Mortem Template — Learn From Every Incident — `post_mortem_template.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q requests redis


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Severity Classification — How Bad Is It?

Not every issue is a 3 AM wake-up call. Classify first, then respond proportionally.

**`incident_classifier.py`** · _Block 1 of 4_

INCIDENT SEVERITY CLASSIFIER — AI-specific severity levels with automated


In [ ]:
# =============================================
# INCIDENT SEVERITY CLASSIFIER
# AI-specific severity levels with automated
# detection and escalation rules.
# =============================================

from enum import Enum
from dataclasses import dataclass, field
from datetime import datetime

class Severity(Enum):
    SEV1 = "sev1"  # Critical: total outage or safety/legal risk
    SEV2 = "sev2"  # Major: significant quality degradation or cost runaway
    SEV3 = "sev3"  # Minor: partial degradation, workaround exists
    SEV4 = "sev4"  # Low: cosmetic, minor quality dip, non-urgent

@dataclass
class Incident:
    title: str
    severity: Severity
    category: str            # "model", "prompt", "data", "infra", "cost", "safety"
    detected_by: str         # "alert", "user_report", "manual", "automated_test"
    description: str
    impact: str
    started_at: str = field(default_factory=lambda: datetime.now().isoformat())
    resolved_at: str = ""
    commander: str = ""
    status: str = "investigating"  # investigating, mitigating, resolved, monitoring

class IncidentClassifier:
    """Classify AI incidents by severity based on signals."""
    
    def classify(self, signals: dict) -> Severity:
        """Auto-classify severity from monitoring signals."""
        
        # ── SEV1: Critical ──
        if signals.get("total_outage"):           return Severity.SEV1
        if signals.get("pii_leak_detected"):       return Severity.SEV1
        if signals.get("safety_violation"):        return Severity.SEV1
        if signals.get("cost_per_hour_usd", 0) > 100: return Severity.SEV1
        
        # ── SEV2: Major ──
        if signals.get("error_rate_pct", 0) > 20:    return Severity.SEV2
        if signals.get("quality_score_drop", 0) > 2: return Severity.SEV2  # 2+ point drop out of 10
        if signals.get("hallucination_rate_pct", 0) > 15: return Severity.SEV2
        if signals.get("cost_per_hour_usd", 0) > 50: return Severity.SEV2
        if signals.get("latency_p99_ms", 0) > 30000:  return Severity.SEV2
        
        # ── SEV3: Minor ──
        if signals.get("error_rate_pct", 0) > 5:     return Severity.SEV3
        if signals.get("quality_score_drop", 0) > 1: return Severity.SEV3
        if signals.get("cache_hit_rate_drop", 0) > 20: return Severity.SEV3
        
        # ── SEV4: Low ──
        return Severity.SEV4
    
    RESPONSE_TIMES = {
        Severity.SEV1: {"acknowledge": "5 min", "mitigate": "30 min", "resolve": "4 hours"},
        Severity.SEV2: {"acknowledge": "15 min", "mitigate": "2 hours", "resolve": "24 hours"},
        Severity.SEV3: {"acknowledge": "1 hour", "mitigate": "1 day", "resolve": "1 week"},
        Severity.SEV4: {"acknowledge": "1 day", "mitigate": "1 week", "resolve": "1 month"},
    }

# ── Demo ──
classifier = IncidentClassifier()

scenarios = [
    {"name": "PII in chatbot response", "pii_leak_detected": True},
    {"name": "Cost spike: $80/hour", "cost_per_hour_usd": 80},
    {"name": "Quality dropped by 1.5 points", "quality_score_drop": 1.5},
    {"name": "Error rate at 8%", "error_rate_pct": 8},
    {"name": "Slightly slower responses", "latency_p99_ms": 5000},
]

print("Incident classification:\n")
for s in scenarios:
    sev = classifier.classify(s)
    times = classifier.RESPONSE_TIMES[sev]
    colors = {"sev1": "🔴", "sev2": "🟠", "sev3": "🟡", "sev4": "🟢"}
    print(f"  {colors[sev.value]} {sev.value}: {s['name']}")
    print(f"     Acknowledge: {times['acknowledge']} | Mitigate: {times['mitigate']} | Resolve: {times['resolve']}")


> **What just happened?** AI-specific severity classification: SEV1 (PII leak, safety violation, total outage, >$100/hour cost — acknowledge in 5 min). SEV2 (>20% error rate, 2+ point quality drop, >15% hallucination rate — acknowledge in 15 min). SEV3 (>5% errors, 1-point quality drop — acknowledge in 1 hour). SEV4 (cosmetic, minor latency — acknowledge in 1 day). Traditional monitoring catches SEV1 (outage). AI monitoring catches SEV2 (quality degradation) — the silent killer.


### Step 2 · Incident Response Playbooks — What to Do for Each Failure Mode

6 AI-specific failure modes, each with a step-by-step response.

**`playbooks.py`** · _Block 2 of 4_

INCIDENT RESPONSE PLAYBOOKS — Step-by-step for 6 AI failure modes.


In [ ]:
# =============================================
# INCIDENT RESPONSE PLAYBOOKS
# Step-by-step for 6 AI failure modes.
# =============================================

PLAYBOOKS = {
    "model_outage": {
        "title": "LLM Provider Outage",
        "severity": "SEV1-SEV2",
        "detection": "Error rate >50%, provider status page shows incident",
        "steps": [
            "1. CHECK: Verify on provider status page (status.cloud.google.com)",
            "2. MITIGATE: Circuit breaker should auto-switch to fallback provider (10.2)",
            "3. VERIFY: Check fallback is working: curl /health",
            "4. COMMUNICATE: Update status page: 'Using backup model, slightly slower'",
            "5. MONITOR: Watch error rate, latency, and cost on fallback",
            "6. RECOVER: When primary returns, verify quality before switching back",
        ],
    },
    "prompt_regression": {
        "title": "Prompt Change Broke Output Quality",
        "severity": "SEV2-SEV3",
        "detection": "Quality score dropped >1 point after deploy, user complaints",
        "steps": [
            "1. IDENTIFY: git log --oneline -5 → find the prompt change",
            "2. COMPARE: Run prompt regression tests on current vs previous version",
            "3. ROLLBACK: gcloud run services update-traffic SERVICE --to-revisions PREV=100",
            "4. VERIFY: Run 10 test queries against rolled-back version",
            "5. ROOT CAUSE: Which test cases did the new prompt fail?",
            "6. FIX: Update prompt, add failing cases to regression suite, redeploy",
        ],
    },
    "cost_runaway": {
        "title": "Cost Spike / Budget Exceeded",
        "severity": "SEV1-SEV2",
        "detection": "Cost alert fires, daily budget exceeded in hours",
        "steps": [
            "1. STOP BLEEDING: Set max-instances=1 to cap new requests",
            "2. IDENTIFY: Check logs → which team/endpoint is generating load?",
            "3. CHECK: Is it a retry loop? DDoS? Legitimate traffic spike?",
            "4. If retry loop: fix the bug causing failures → retries → more failures",
            "5. If traffic spike: enable queue-based scaling (10.7), throttle gracefully",
            "6. If DDoS: block the source IP/key, tighten rate limits",
            "7. RECOVER: Gradually restore max-instances, monitor cost",
        ],
    },
    "hallucination_spike": {
        "title": "Hallucination Rate Increased",
        "severity": "SEV2",
        "detection": "Quality monitor shows >15% hallucination markers in output",
        "steps": [
            "1. VERIFY: Sample 20 recent responses — manually check for hallucinations",
            "2. CHECK MODEL: Did the provider update the model version? (changelog)",
            "3. CHECK RAG: If using RAG, check retrieval quality — are chunks relevant?",
            "4. CHECK CACHE: Is the semantic cache returning stale/wrong matches?",
            "5. MITIGATE: Lower temperature, add 'only answer from provided context'",
            "6. If model update: pin to previous model version if API supports it",
            "7. If RAG: re-index documents, check embedding model hasn't changed",
        ],
    },
    "pii_leak": {
        "title": "PII Detected in AI Output",
        "severity": "SEV1",
        "detection": "Output guardrail flagged PII in response, user report",
        "steps": [
            "1. STOP: If systematic, take the endpoint offline immediately",
            "2. SCOPE: How many responses contained PII? Query logs for PII patterns",
            "3. SOURCE: Is PII coming from RAG context? Prompt injection? Training data?",
            "4. PURGE CACHE: Clear all cached responses (they may contain PII)",
            "5. FIX: Add output PII filter (9.11), block PII in RAG retrieval",
            "6. LEGAL: Notify compliance team, assess data breach reporting requirements",
            "7. VERIFY: Run PII scanner on 100 test queries before restoring service",
        ],
    },
    "cache_poisoning": {
        "title": "Cache Serving Wrong / Stale Responses",
        "severity": "SEV2-SEV3",
        "detection": "Users report outdated info, semantic cache matching wrong queries",
        "steps": [
            "1. VERIFY: Compare cached response vs fresh LLM call for same query",
            "2. If stale: Flush the entire cache — redis-cli FLUSHDB",
            "3. If wrong matches: Lower semantic cache threshold from 0.92 to 0.95",
            "4. CHECK TTL: Are TTL policies appropriate? (volatile content cached too long?)",
            "5. WARM: Re-populate cache with known-good responses for top queries",
            "6. PREVENT: Add cache versioning — include model version in cache key",
        ],
    },
}

# ── Print playbooks ──
print("AI Incident Playbooks:\n")
for key, pb in PLAYBOOKS.items():
    print(f"  📋 {pb['title']} [{pb['severity']}]")
    print(f"     Detection: {pb['detection']}")
    print(f"     Steps: {len(pb['steps'])}\n")


> **What just happened?** Six playbooks for AI-specific failure modes: model outage (circuit breaker → fallback), prompt regression (git log → compare → rollback traffic), cost runaway (cap instances → identify source → fix retry loops), hallucination spike (check model version, RAG quality, cache), PII leak (take offline → scope → purge cache → legal), cache poisoning (flush → lower threshold → re-warm). Each playbook is a numbered sequence you can follow at 3 AM without thinking.


### Step 3 · Rollback Strategies — Coded Commands for Every Layer

Five rollback targets: traffic, model, prompt, cache, and full service.

**`rollback.sh`** · _Block 3 of 4_

!/bin/bash — ═══════════════════════════════════════════


In [ ]:
#!/bin/bash
# ═══════════════════════════════════════════
# ROLLBACK STRATEGIES FOR AI SERVICES
# 5 levels: traffic, model, prompt, cache, full
# ═══════════════════════════════════════════

SERVICE="ai-service"
REGION="asia-south1"

# ── ROLLBACK 1: Traffic (instant, zero downtime) ──
# Send 100% traffic back to the previous revision.
# Use when: new deploy caused quality/error regression.
rollback_traffic() {
  echo "🔙 Rolling back traffic to previous revision..."
  PREV_REV=$(gcloud run revisions list --service ${SERVICE} \
    --region ${REGION} --format "value(name)" --limit 2 | tail -1)
  gcloud run services update-traffic ${SERVICE} \
    --region ${REGION} --to-revisions ${PREV_REV}=100
  echo "✅ Traffic now 100% on ${PREV_REV}"
}

# ── ROLLBACK 2: Model (switch to safer model) ──
# Use when: current model hallucinating or producing bad output.
rollback_model() {
  echo "🔙 Switching to fallback model..."
  gcloud run services update ${SERVICE} \
    --region ${REGION} \
    --set-env-vars "DEFAULT_MODEL=gemini-2.0-flash,FALLBACK_MODE=true"
  echo "✅ Switched to gemini-2.0-flash (safe fallback)"
}

# ── ROLLBACK 3: Cache (purge potentially poisoned cache) ──
# Use when: cache returning wrong/stale/PII-containing responses.
rollback_cache() {
  echo "🔙 Flushing AI response cache..."
  # If using Redis/Memorystore:
  redis-cli -h ${REDIS_HOST} -p 6379 FLUSHDB
  # If using in-memory: restart all instances
  gcloud run services update ${SERVICE} --region ${REGION} \
    --set-env-vars "CACHE_VERSION=$(date +%s)"
  echo "✅ Cache flushed. Responses will be regenerated fresh."
}

# ── ROLLBACK 4: Scale down (stop the bleeding on cost) ──
# Use when: cost runaway, DDoS, retry storm.
rollback_scale() {
  echo "🔙 Scaling down to minimum..."
  gcloud run services update ${SERVICE} \
    --region ${REGION} \
    --max-instances 1 \
    --concurrency 5
  echo "✅ Scaled to 1 instance, 5 concurrent. Bleeding stopped."
}

# ── ROLLBACK 5: Full (take service offline) ──
# Use when: PII leak, safety violation, legal risk.
# NUCLEAR OPTION: serve static error page instead.
rollback_full() {
  echo "🔙 TAKING SERVICE OFFLINE..."
  gcloud run services update-traffic ${SERVICE} \
    --region ${REGION} --to-revisions maintenance-page=100
  echo "⛔ Service offline. Maintenance page serving."
  echo "   Remember to notify stakeholders and update status page."
}

# ── Usage ──
echo "Rollback commands ready:"
echo "  rollback_traffic  — revert to previous deploy"
echo "  rollback_model    — switch to safe fallback model"
echo "  rollback_cache    — purge all cached responses"
echo "  rollback_scale    — scale to minimum (stop cost)"
echo "  rollback_full     — take service completely offline"


> **What just happened?** Five rollback levels, escalating in severity: Traffic (instant — shift 100% to previous revision, zero downtime). Model (switch to a safer/cheaper fallback model via env var). Cache (flush Redis + bump cache version to force regeneration). Scale (cap at 1 instance, 5 concurrent — stops cost hemorrhage). Full (take offline, serve maintenance page — nuclear option for PII/safety). Keep these as shell functions in your ops runbook. At 3 AM, you don't write gcloud commands from memory.


### Step 4 · Post-Mortem Template — Learn From Every Incident

Structured format with AI-specific root cause categories. Blameless. Action-item focused.

**`post_mortem_template.py`** · _Block 4 of 4_

POST-MORTEM TEMPLATE FOR AI INCIDENTS — Fill this out within 48 hours of resolution.


In [ ]:
# =============================================
# POST-MORTEM TEMPLATE FOR AI INCIDENTS
# Fill this out within 48 hours of resolution.
# Blameless. Focus on systems, not people.
# =============================================

POST_MORTEM_TEMPLATE = """
# Post-Mortem: {title}

**Date:** {date}
**Severity:** {severity}
**Duration:** {start_time} → {end_time} ({duration_minutes} min)
**Author:** {author}
**Status:** Resolved

---

## Summary
{one_paragraph_summary}

## Impact
- **Users affected:** {users_affected}
- **Requests affected:** {requests_affected}
- **Financial impact:** {financial_impact}
- **Reputational impact:** {reputational_impact}

## Timeline (all times IST)
| Time | Event |
|------|-------|
| {t1} | Alert fired: {alert_description} |
| {t2} | Incident commander assigned: {commander} |
| {t3} | Root cause identified: {root_cause_short} |
| {t4} | Mitigation applied: {mitigation_action} |
| {t5} | Service fully restored |
| {t6} | Monitoring confirms resolution |

## Root Cause

### Category: {root_cause_category}

AI Root Cause Categories:
- MODEL:  Provider model update, model degradation, rate limit
- PROMPT: Prompt change regression, prompt injection exploit
- DATA:   RAG data quality, stale embeddings, corrupted index
- CACHE:  Cache poisoning, wrong semantic matches, stale TTL
- INFRA:  Outage, scaling failure, deployment error
- COST:   Retry storm, missing rate limits, model misrouting
- SAFETY: PII leak, harmful output, bias amplification

### Detailed Explanation
{root_cause_detailed}

## Detection
- **How detected:** {detection_method}
- **Time to detect:** {time_to_detect}
- **Why not detected sooner:** {detection_gap}

## Mitigation
{mitigation_steps}

## Action Items
| # | Action | Owner | Due | Priority |
|---|--------|-------|-----|----------|
| 1 | {action_1} | {owner_1} | {due_1} | P0 |
| 2 | {action_2} | {owner_2} | {due_2} | P1 |
| 3 | {action_3} | {owner_3} | {due_3} | P1 |
| 4 | {action_4} | {owner_4} | {due_4} | P2 |

## Lessons Learned
### What went well
{what_went_well}

### What went wrong
{what_went_wrong}

### Where we got lucky
{where_lucky}
"""

# ── Example filled post-mortem ──
print("""
Example Post-Mortem: Hallucination Spike After Model Update

Summary:
  On March 15, Gemini 2.0 Flash received a minor version update.
  Our hallucination monitoring detected a jump from 3% to 18% in
  responses containing factual inaccuracies. 450 users received
  responses with incorrect course pricing before mitigation.

Root Cause: MODEL
  Google updated gemini-2.0-flash from version 002 to 003.
  The new version changed how the model handles numerical
  facts from context windows, causing it to generate plausible
  but incorrect numbers (prices, dates) at a higher rate.

Action Items:
  P0: Pin model version in deployment config (not just model name)
  P0: Add numerical fact-checking to output validator
  P1: Set up model version change alerts from provider changelogs
  P2: Build A/B test pipeline for model version upgrades
""")


> **What just happened?** A structured post-mortem template with 7 AI-specific root cause categories (MODEL, PROMPT, DATA, CACHE, INFRA, COST, SAFETY). The example shows a real scenario: Gemini updated its model, hallucination rate jumped from 3% to 18%, 450 users affected. The root cause analysis identifies the specific change (version 002 → 003, numerical fact handling). Action items are prioritized P0-P2 with owners and deadlines. "Where we got lucky" is the most important section — it reveals near-misses that could have been worse.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
